# 旅行商问题（TSP）：一个模型一个 Notebook

## 这个模型解决什么问题
从一个仓库出发，把每个客户点都走一遍（各一次），最后回到仓库，让总路程最短。
外卖配送、巡检路线、灾后物资投送都是这个模型。

## 两种解法，按规模选

| 方法 | 适用 | 特点 |
| --- | --- | --- |
| **MILP 精确解**（MTZ 子回路消除） | 点数 ≤ 12～15 | 一定得到全局最优，但变量数随点数平方增长 |
| **最近邻 + 2-opt 启发式** | 点数几十到几百 | 快，通常离最优 2%～5%，不保证最优 |

MTZ 约束：`u_i - u_j + n·x_ij <= n - 1`，用顺序变量 u 把子回路切断。

## 两种数据输入方式
| 方式 | 数据长什么样 |
| --- | --- |
| 已处理矩阵 | n×n 距离矩阵（第一列是节点名） |
| 原始表（坐标） | 每个点的 x / y 坐标，模块内自动算欧氏距离 |

## 目录
0 环境 / 1 配置 / 2 数据接口 / 3 求解（精确 + 启发式）/ 4 距离矩阵示例 / 5 坐标表示例 / 6 导出

> 依赖 `numpy pandas scipy`；精确解用 `scipy.optimize.milp`。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    data_path=DATA_DIR / "优化_配送距离.csv",   # 距离矩阵；换题目改这里
    start="仓库",        # 起点名称；用矩阵时必须是索引里存在的名字
    method="auto",       # auto=点数≤12 用精确解，否则启发式；也可强制 exact / heuristic
    tag="配送路径",
    make_plots=True,
    save_excel=True,
)
print("配置：", CFG)


配置： {'data_path': WindowsPath('D:/git项目/导入数据/优化_配送距离.csv'), 'start': '仓库', 'method': 'auto', 'tag': '配送路径', 'make_plots': True, 'save_excel': True}


## 2. 数据接口


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 求解：精确 MILP 与启发式

点数少用精确解，点数多用启发式，`solve_tsp` 会按规模自动选。


In [4]:
# =====================================================================
# 3. 求解
# =====================================================================
def tsp_exact(D, start=0, time_limit=60):
    """MILP 精确解（MTZ 子回路消除）。

    变量：x_ij（i≠j 表示走 i→j）+ u_i（访问顺序，用于切断子回路）
    约束：每个点出度 1、入度 1；u_i - u_j + n·x_ij <= n-1
    """
    n = D.shape[0]
    pairs = [(i, j) for i in range(n) for j in range(n) if i != j]
    idx = {p: k for k, p in enumerate(pairs)}
    nv = len(pairs) + n

    c = np.zeros(nv)
    for (i, j), k in idx.items():
        c[k] = D[i, j]

    A_eq, b_eq = [], []
    for i in range(n):                                  # 出度 = 1
        row = np.zeros(nv); row[[idx[(i, j)] for j in range(n) if j != i]] = 1
        A_eq.append(row); b_eq.append(1)
    for j in range(n):                                  # 入度 = 1
        row = np.zeros(nv); row[[idx[(i, j)] for i in range(n) if i != j]] = 1
        A_eq.append(row); b_eq.append(1)

    A_ub, b_ub = [], []
    for i in range(1, n):
        for j in range(1, n):
            if i == j:
                continue
            row = np.zeros(nv)
            row[idx[(i, j)]] = n
            row[len(pairs) + i] = 1
            row[len(pairs) + j] = -1
            A_ub.append(row); b_ub.append(n - 1)

    bounds = [(0, 1)] * len(pairs) + [(0, n - 1)] * n
    lb = np.array([b[0] for b in bounds], dtype=float)
    ub = np.array([b[1] for b in bounds], dtype=float)
    cons = [LinearConstraint(np.array(A_eq), b_eq, b_eq),
            LinearConstraint(np.array(A_ub), -np.inf, np.array(b_ub))]
    r = sp_opt.milp(c, integrality=np.ones(nv, dtype=int),
                    bounds=Bounds(lb, ub), constraints=cons,
                    options={"time_limit": time_limit})
    if r.x is None:
        return None, np.inf
    succ = {i: j for (i, j), k in idx.items() if r.x[k] > 0.5}
    route = [start]
    for _ in range(n - 1):
        route.append(succ[route[-1]])
    route.append(start)
    return route, float(r.fun)


def tsp_heuristic(D, start=0):
    """最近邻构造 + 2-opt 改进。

    2-opt 的思路：如果两条边交叉（a→b 接 c→d 比 a→c 接 b→d 更长），
    就把中间那段路线反转。对称距离矩阵才启用。
    """
    n = D.shape[0]
    unvisited = set(range(n)) - {start}
    route = [start]
    while unvisited:
        nxt = min(unvisited, key=lambda j: D[route[-1], j])
        route.append(nxt)
        unvisited.remove(nxt)
    route.append(start)

    if np.allclose(D, D.T):                             # 对称才做 2-opt
        improved = True
        while improved:
            improved = False
            for i in range(1, len(route) - 2):
                for k in range(i + 1, len(route) - 1):
                    a, b = route[i - 1], route[i]
                    c_, d = route[k], route[k + 1]
                    if D[a, b] + D[c_, d] > D[a, c_] + D[b, d] + 1e-9:
                        route[i:k + 1] = reversed(route[i:k + 1])
                        improved = True
    total = sum(D[a, b] for a, b in zip(route[:-1], route[1:]))
    return route, float(total)


def solve_tsp(D, names, start="auto", method="auto"):
    """统一入口：按点数与方法选择精确解还是启发式。"""
    if isinstance(start, str) and start != "auto":
        start_idx = names.index(start)
    elif start == "auto":
        start_idx = 0
    else:
        start_idx = int(start)
    n = len(names)
    if method == "auto":
        method = "exact" if n <= 12 else "heuristic"
    t0 = time.time()
    if method == "exact":
        route, total = tsp_exact(D, start_idx)
        if route is None:                               # 精确解失败就退回启发式
            route, total = tsp_heuristic(D, start_idx)
            return route, total, "heuristic", time.time() - t0
        return route, total, "exact", time.time() - t0
    route, total = tsp_heuristic(D, start_idx)
    return route, total, "heuristic", time.time() - t0


def route_table(route, D, names):
    """路线明细表：第几站、节点、到下一站的距离。"""
    rows = []
    for k, (a, b) in enumerate(zip(route[:-1], route[1:]), start=1):
        rows.append({"顺序": k, "从": names[a], "到": names[b],
                     "距离": round(D[a, b], 3)})
    rows.append({"顺序": len(route), "从": names[route[-1]], "到": "—（回到起点）",
                 "距离": 0.0})
    return pd.DataFrame(rows)


## 4. 示例 A：距离矩阵


In [5]:
# =====================================================================
# 4. 示例 A：距离矩阵（已处理好，最常见）
# =====================================================================
dist = load_matrix(CFG["data_path"])
names = list(dist.index)
D = dist.to_numpy(dtype=float)
print(f"距离矩阵：{D.shape[0]} 个节点")
print(dist.round(1).to_string())

route, total, used, elapsed = solve_tsp(D, names, CFG["start"], CFG["method"])
print(f"\n=== 最优路线（{used}，用时 {elapsed:.3f}s）===")
print(" → ".join(names[i] for i in route))
print(f"总路程：{total:.3f}")
print()
print(route_table(route, D, names).to_string(index=False))

# 核对：每个点恰好访问一次、首尾都是起点
visited = [names[i] for i in route[:-1]]
print("\n访问点数：", len(set(visited)), "/", len(names),
      "| 各访问一次：", bool(len(set(visited)) == len(names) == len(visited)),
      "| 回到起点：", names[route[0]] == names[route[-1]])


距离矩阵：8 个节点
      仓库    C1    C2    C3    C4    C5    C6    C7
仓库   0.0  27.2  86.6  34.7  64.6  63.3  40.5  39.2
C1  27.2   0.0  81.3  13.2  77.1  53.9  24.9  62.7
C2  86.6  81.3   0.0  69.3  52.6  28.1  57.1  82.6
C3  34.7  13.2  69.3   0.0  71.7  41.5  12.3  64.3
C4  64.6  77.1  52.6  71.7   0.0  53.5  63.6  38.6
C5  63.3  53.9  28.1  41.5  53.5   0.0  29.2  70.3
C6  40.5  24.9  57.1  12.3  63.6  29.2   0.0  62.8
C7  39.2  62.7  82.6  64.3  38.6  70.3  62.8   0.0

=== 最优路线（exact，用时 0.050s）===
仓库 → C1 → C3 → C6 → C5 → C2 → C4 → C7 → 仓库
总路程：240.400

 顺序  从       到   距离
  1 仓库      C1 27.2
  2 C1      C3 13.2
  3 C3      C6 12.3
  4 C6      C5 29.2
  5 C5      C2 28.1
  6 C2      C4 52.6
  7 C4      C7 38.6
  8 C7      仓库 39.2
  9 仓库 —（回到起点）  0.0

访问点数： 8 / 8 | 各访问一次： True | 回到起点： True


## 5. 示例 B：坐标表 → 距离矩阵


In [6]:
# =====================================================================
# 5. 示例 B：坐标表 → 自动算距离（原始数据模式）
# =====================================================================
# 附件只给经纬度/平面坐标时，先算距离矩阵再套同一套求解函数。
coord = pd.DataFrame({
    "节点": ["仓库", "C1", "C2", "C3", "C4", "C5", "C6", "C7"],
    "x": [10, 82, 35, 91, 20, 60, 75, 48],
    "y": [50, 12, 78, 60, 15, 90, 35, 5],
}).set_index("节点")
print("坐标：")
print(coord.to_string())

# 欧氏距离；如果是经纬度，把下面的公式换成 haversine（见文末说明）
diff = coord.to_numpy()[:, None, :] - coord.to_numpy()[None, :, :]
D_b = np.sqrt((diff ** 2).sum(axis=-1))
D_b = np.round(D_b, 2)
names_b = list(coord.index)
print(f"\n自动生成距离矩阵：{D_b.shape}")

route_b, total_b, used_b, t_b = solve_tsp(D_b, names_b, "仓库", CFG["method"])
print(f"\n=== 路线（{used_b}，{t_b:.3f}s）===")
print(" → ".join(names_b[i] for i in route_b))
print(f"总路程：{total_b:.2f}")
print()
print(route_table(route_b, D_b, names_b).to_string(index=False))

# 顺便给个最优性参照：精确解在这 8 个点上几十毫秒就能跑完
r_ex, t_ex = tsp_exact(D_b, 0)
if r_ex:
    print(f"\n精确解总距离 {t_ex:.2f} | 启发式 {total_b:.2f} | "
          f"差距 {(total_b - t_ex) / t_ex * 100:.2f}%")


坐标：
     x   y
节点        
仓库  10  50
C1  82  12
C2  35  78
C3  91  60
C4  20  15
C5  60  90
C6  75  35
C7  48   5

自动生成距离矩阵：(8, 8)

=== 路线（exact，0.071s）===
仓库 → C4 → C7 → C1 → C6 → C3 → C5 → C2 → 仓库
总路程：262.97

 顺序  从       到    距离
  1 仓库      C4 36.40
  2 C4      C7 29.73
  3 C7      C1 34.71
  4 C1      C6 24.04
  5 C6      C3 29.68
  6 C3      C5 43.14
  7 C5      C2 27.73
  8 C2      仓库 37.54
  9 仓库 —（回到起点）  0.00

精确解总距离 262.97 | 启发式 262.97 | 差距 0.00%


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出与路线图
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL:
    # 用 MDS 之类做真实布局成本高，这里用坐标表示例的位置来画，
    # 只表达"谁接了谁"，不表达真实地理方位。
    pos = {n: coord.loc[n].to_numpy() for n in names} if "coord" in dir() else None
    if pos is None:                      # 距离矩阵模式：用环形布局代替
        theta = np.linspace(0, 2 * np.pi, len(names), endpoint=False)
        pos = {n: np.array([np.cos(t), np.sin(t)]) for n, t in zip(names, theta)}
    fig, ax = plt.subplots(figsize=(6, 5))
    for a, b in zip(route[:-1], route[1:]):
        ax.annotate("", xy=pos[names[b]], xytext=pos[names[a]],
                    arrowprops=dict(arrowstyle="->", color="#4a7fd8", lw=1.5))
    for n in names:
        ax.scatter(*pos[n], s=70, color="#d95f5f" if n == names[route[0]] else "#333", zorder=3)
        ax.annotate(n, pos[n], textcoords="offset points", xytext=(6, 6), fontsize=9)
    ax.set_title(f"配送路线（总路程 {total:.2f}）")
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()

written = export_result(
    {"结果摘要": pd.DataFrame([{"总路程": round(total, 3), "节点数": len(names),
                              "解法": used, "用时(秒)": round(elapsed, 3)}]),
     "路线明细": route_table(route, D, names),
     "距离矩阵": dist.reset_index()},
    CFG, fig=fig, fig_name="路线图")
print(written)


[表格] 配送路径_结果汇总.xlsx（3 个 Sheet）
[图片] 配送路径_路线图.png
[输出]  D:\git项目\输出数据\优化_配送路径
{'主表': WindowsPath('D:/git项目/输出数据/优化_配送路径/配送路径_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_配送路径/配送路径_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_配送路径/配送路径_路线图.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。

**坐标是经纬度时**：把欧氏距离换成 haversine 公式，`R=6371` 公里，`d=2R·asin(√(sin²(Δφ/2)+cosφ1·cosφ2·sin²(Δλ/2)))`。
